In [ ]:
import os
import pandas as pd
import time
import json

INPUT_CSV = "../generated_data/user_profiles_fixed.csv"
OUTPUT_CSV = "../generated_data/more_user_profiles_with_imgdesc.csv"
CHECKPOINT_FILE = "checkpoint.json"

def load_checkpoint():
    """Load checkpointed row index if exists."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return json.load(f).get("last_completed_index", -1)
    return -1

def save_checkpoint(index):
    """Save last successfully processed row index."""
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"last_completed_index": index}, f)

def process_profiles():
    df = pd.read_csv(INPUT_CSV)

    # Make sure the column exists even if we resume
    if "profile_image_description" not in df.columns:
        df["profile_image_description"] = None

    last_completed_index = load_checkpoint()
    print(f"Resuming from row {last_completed_index + 1}")

    for idx in range(last_completed_index + 1, len(df)):
        row = df.iloc[idx]
        image_url = row.get("profile_image_url")

        print(f"[{idx+1}/{len(df)}] Processing user_id={row.get('id', idx)}")

        if pd.isna(image_url) or not str(image_url).startswith("http"):
            desc = "Profile image description is not available for this user"
        else:
            try:
                result = send_prompt(
                    prompt="""
This is a social media user's profile image.

1. List all visible physical characteristics of any people (skin tone, hairstyle, facial hair, wrinkles, clothing, etc.).

2. List all visible contextual or lifestyle cues in the image (e.g., objects, backgrounds, locations, clothing style, accessories, logos, symbols).

3. Based only on those visible cues, summarize possible demographic information only if it can be inferred with reasonable certainty.:
   - Approximate age range
   - Gender presentation
   - Racial or ethnic background
   - Education level (only if directly indicated, e.g., graduation attire)
   - Income level (only if strongly suggested by visible lifestyle cues)
   - Marital status (only if directly visible, e.g., wedding ring or couple photo)
   - Partisan or political affiliation (only if explicit logos, signs, or symbols are visible)

If uncertain or not visible, explicitly state that the information is unclear.

4. Speculation Score: For each demographic attribute, provide a speculation score using the following scale:

0–20 (Low speculation): The image provides clear and direct information relevant to the attribute (e.g., explicit visual evidence in the profile image).

21–40 (Moderate-low speculation): The image provides indirect but strong indicators relevant to the attribute.

41–60 (Moderate speculation): The image provides some hints or partial information relevant to the attribute.

61–80 (Moderate-high speculation): The image provides limited and weak indicators relevant to the attribute.

81–100 (High speculation): The image provides no or almost no information relevant to the attribute.

5. Explanation: For each speculation score, briefly explain why you assigned that score, referring to visible cues or lack thereof.
                    """,
                    image_url=image_url,
                    azure_model_name="Llama-4-Maverick-17B-128E-Instruct-FP8"
                )
                desc = result["choices"][0]["message"]["content"]
            except Exception as e:
                print(f"Error processing image: {e}")
                desc = "Profile image description is not available for this user"

        df.at[idx, "profile_image_description"] = desc

        # Save partial results every row 
        df.to_csv(OUTPUT_CSV, index=False)
        save_checkpoint(idx)
        time.sleep(1)  # avoid API rate limits

    print("Processing complete.")
    # Remove checkpoint when done
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

if __name__ == "__main__":
    process_profiles()


In [2]:
import pandas as pd

In [3]:
merged_df = pd.read_csv("../generated_data/user_profiles_fixed.csv")

In [4]:
test_profile = merged_df.iloc[48:50].copy()


In [5]:
test_profile

,Unnamed: 0,gpt_id,id,created_at.users,description,location,protected,verified_type,username,public_metrics.followers_count,...,gpt_speculation_QUOTA_PARTISAN LOYALTIES,"gpt_speculation_QUOTA_IDEOLOGICALLY, THIS PERSON APPEARS TO BE",gpt_speculation_QUOTA_PAST VOTE - TURNOUT IN THE 2020 PRESIDENTIAL ELECTION,gpt_speculation_QUOTA_PAST VOTE - TURNOUT IN THE 2022 HOUSE OF REPRESENTATIVES ELECTION,gpt_speculation_QUOTA_PAST VOTE - VOTE CHOICE IN THE 2020 PRESIDENTIAL ELECTION,gpt_speculation_QUOTA_PAST VOTE - VOTE CHOICE IN THE 2022 HOUSE OF REPRESENTATIVES ELECTION,clean...pool,clean...pool.date,clean...state_simple,image_accessible
48,105758,0f06ae1bd1c0037cbcd5f03b343e915068408395224557...,1700888571140599808,2023-09-10T15:08:14.000Z,Nature lover. Christian Astrologer. QM researc...,Best Planet in Solar System,False,blue,MerMadrigal,583,...,50,40,30.0,30,60,50,democracy,2024-09-24T16:20:27Z,USA,True
49,107149,0f1469d11a8fd52a5763e124824e8973399bb74cd2a6a4...,1707747985294274560,2023-09-29T13:24:18.000Z,Retired USMC and Dept. of Veterans Affairs. I ...,FL,False,none,rec165900,1992,...,40,30,30.0,30,40,50,protectionism,2024-10-15T05:31:31Z,Florida,True


In [11]:
import os
import requests
import json
import time
import re
import random
import unicodedata
import pandas as pd
import tempfile
import base64
import mimetypes

# function to normalize text for parsing
def normalize_text(text):
    # Normalize unicode characters, remove markdown formatting, lowercase, and strip dashes
    if not text:
        return text
    
    text = str(text)
    text = text.replace('*', '').replace('<<', '').replace('>>', '')
    text = re.sub(r'[-\u2010-\u2015\u2212\uFE58\uFE63\uFF0D]', '', text)
    normalized = unicodedata.normalize('NFKD', text)
    ascii_text = normalized.encode('ASCII', 'ignore').decode('ASCII')

    # Clean up whitespace + lowercase
    return ' '.join(ascii_text.split()).lower()

# ------------------------------------------------------------
# 0. load checkpoint
# ------------------------------------------------------------

checkpoint_file = "../generated_data/image_demographic_extraction/checkpoint.json"

def load_checkpoint():
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"last_index": -1} 

def save_checkpoint(last_index):
    os.makedirs(os.path.dirname(checkpoint_file), exist_ok=True)
    with open(checkpoint_file, "w", encoding="utf-8") as f:
        json.dump({"last_index": last_index}, f)

# ------------------------------------------------------------
# 1. Image processing function
# ------------------------------------------------------------

def send_prompt(
    system=None,
    prompt=None,
    image_url=None,             # if set, we download to a temp file and inline it
    image_path=None,            # local file path (sent as data URL)
    image_detail="auto",        
    seed=None,
    azure_api_key=os.getenv("AZURE_API_KEY"),
    azure_chat_url=None,        # built if not provided
    azure_model_name=os.getenv("AZURE_MODEL_NAME"),
    max_completion_tokens=1024,
    temperature=float(os.getenv("TEMPERATURE", "1.0")),
    top_p=float(os.getenv("TOP_P", "1.0")),
    frequency_penalty=float(os.getenv("FREQUENCY_PENALTY", "0.0")),
    presence_penalty=float(os.getenv("PRESENCE_PENALTY", "0.0")),
    max_time=int(os.getenv("MAX_TIME", "30"))
):

    # --- default azure_chat_url if not supplied ---
    if azure_chat_url is None:
        azure_endpoint_base = "https://aipolling.services.ai.azure.com/models"
        azure_api_version  = "2024-05-01-preview"
        azure_chat_url = f"{azure_endpoint_base}/chat/completions?api-version={azure_api_version}"

    # --- helpers ---
    def guess_mime_from_ext(path):
        """Guess MIME type from file extension, fallback to image/jpeg."""
        mime, _ = mimetypes.guess_type(path)
        return mime if mime else "image/jpeg"

    def image_to_data_url(path, mime=None):
        """Read file, encode as base64, return as data URL."""
        if not os.path.exists(path):
            raise FileNotFoundError(f"image_path not found: {path}")
        if mime is None:
            mime = guess_mime_from_ext(path)
        with open(path, "rb") as f:
            encoded = base64.b64encode(f.read()).decode("utf-8")
        return f"data:{mime};base64,{encoded}"

    tmp_path = None
    # If image_url is provided, download it
    if image_url:
        ext = os.path.splitext(image_url)[1].lower()
        suffix = ext if ext else ".bin"
        tmp_fd, tmp_path = tempfile.mkstemp(suffix=suffix)
        os.close(tmp_fd)  # we'll reopen with write
        resp_img = requests.get(image_url, timeout=max_time)
        if resp_img.status_code >= 400:
            raise RuntimeError(f"Failed to download image_url (HTTP {resp_img.status_code}): {image_url}")
        with open(tmp_path, "wb") as f:
            f.write(resp_img.content)
        image_path = tmp_path

    # --- build messages ---
    messages_list = []
    if system:
        messages_list.append({"role": "system", "content": system})

    user_content = [{"type": "text", "text": prompt}]

    if image_path:
        mime = guess_mime_from_ext(image_path)
        data_url = image_to_data_url(image_path, mime)
        user_content.append({
            "type": "image_url",
            "image_url": {"url": data_url, "detail": image_detail}
        })

    messages_list.append({"role": "user", "content": user_content})

    # --- body payload ---
    body = {
        "model": azure_model_name,
        "messages": messages_list,
        "temperature": float(temperature),
        "top_p": float(top_p),
        "frequency_penalty": float(frequency_penalty),
        "presence_penalty": float(presence_penalty),
        "max_completion_tokens": int(max_completion_tokens),
        "max_tokens": int(max_completion_tokens),  # compat
        "seed": int(seed) if seed is not None else None
    }

    # --- request ---
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
        "api-key": azure_api_key
    }

    resp = requests.post(
        azure_chat_url,
        headers=headers,
        json=body,
        timeout=max_time
    )

    raw_content = resp.text
    if resp.status_code != 200:
        raise RuntimeError(f"API request failed with status {resp.status_code}, body: {raw_content}")

    result = resp.json()

    # --- cleanup tmp file if used ---
    if tmp_path and os.path.exists(tmp_path):
        try:
            os.remove(tmp_path)
        except Exception:
            pass

    return result

# ------------------------------------------------------------
# 2. Load demographic categories from JSON
# ------------------------------------------------------------
with open("../generated_data/image_belief_features.json", "r", encoding="utf-8") as f:
    feature_data = json.load(f)

demo_features = feature_data["demo"]

# Normalize titles for the matching
for demo in demo_features:
    demo["title"] = normalize_text(demo["title"])

# ------------------------------------------------------------
# 3. Image prompt for demographic features
# ------------------------------------------------------------
image_prompt_template = """
I will show you a social media user's profile image.

Based SOLELY on this profile image, I will show you a number of demographic categories to which this user may belong to.
The categories are preceded by a title (e.g. "AGE:" or "SEX:" etc.) and a symbol (e.g. "A1", "A2" or "E1" etc.).
Please select, for each title, the most likely category to which this user belongs to BASED SOLELY ON THE VISUAL CONTENT OF THE PROFILE IMAGE.

In your answer present, for each title, the selected symbol.
Write out in full the category associated with the selected symbol.
The chosen symbol / category must be the most likely to accurately represent this user.
You must only select one symbol / category per title.
A title, symbol and category cannot appear more than once in your answer.

For each selected symbol / category, please note the level of Speculation involved in this selection.
Present the Speculation level for each selection on a scale from 0 (not speculative at all, clear visual evidence in the profile image) to 100 (fully speculative, no visual information related to this title in the profile image).
Speculation levels should be a direct measure of the amount of useful visual information available in the profile image.
Speculation levels pertain only to the visual information available in the profile image and should not be affected by additional information available to you from any other source.


To ensure consistency, use the following guidelines to determine speculation levels:

0-20 (Low speculation): The profile image provides clear and direct visual information relevant to the title (e.g., explicit visual evidence of age, gender presentation, etc.)
21-40 (Moderate-low speculation): The profile image provides indirect but strong visual indicators relevant to the title.
41-60 (Moderate speculation): The profile image provides some visual hints or partial information relevant to the title.
61-80 (Moderate-high speculation): The profile image provides limited and weak visual indicators relevant to the title.
81-100 (High speculation): The profile image provides no or almost no visual information relevant to the title.

For each selected category, please explain at length what visual features of the profile image contributed to your choice and your speculation level.

Preserve a strictly structured answer to ease parsing of the text.
Format your output as follows (this is just an example):

title: AGE
explanation: ...
symbol: A1)
category: 18-25
speculation: 90

For every title, you must include exactly these 5 fields: title, explanation, symbol, category, speculation. Do not omit any of them.
Please keep the EXACT WORDING for the titles!
YOU MUST GIVE AN ANSWER FOR EVERY TITLE!
"""

# ------------------------------------------------------------
# 4. Build dynamic image demographic prompt
# ------------------------------------------------------------
def build_image_demo_prompt(demo_features):
    
    # randomly shuffle the demographics to prevent position bias
    shuffled_demo = demo_features[:]
    random.shuffle(shuffled_demo)

    categories_text = "\n".join(
        [
            f"- **{demo['title']}:** \n    Choose one: [\n    " +
            ", \n    ".join([f"{key}) {val}" for key, val in demo["options"].items()]) +
            "]"
            for demo in shuffled_demo
        ]
    )

    return image_prompt_template + "\n\nBelow is the list of demographic categories to which this user may belong to:\n" + categories_text

# ----------------------------
# 5. Result Parsing Function
# ----------------------------

def parse_demo_response(response, demo_features, prefix="Image_Demo"):
    categories = {}
    explanations = {}
    speculations = {}
    symbols = {}

    for demo in demo_features:
        safe_key = re.sub(r"\W+", "_", demo["title"])
        if prefix:
            safe_key = f"{prefix}_{safe_key}"
        categories[f"{safe_key}_Category"] = None
        explanations[f"{safe_key}_Explanation"] = None
        symbols[f"{safe_key}_Symbol"] = None
        speculations[f"{safe_key}_Speculation"] = None

    demo_sections = re.split(r"(?=title:\s*)", str(response), flags=re.IGNORECASE)
    demo_sections = [section.strip() for section in demo_sections if section.strip() and not section.lower().startswith('below')]
    print(f"[DEBUG] Found {len(demo_sections)} sections to parse") 
    
    # Pattern for individual demographic
    pattern = (
        r"title:?\s*(.*?)\s*[\r\n]"
        r"explanation:?\s*(.*?)\s*[\r\n]"
        r"symbol:?\s*(.*?)\s*[\r\n]"
        r"category:?\s*(.*?)\s*[\r\n]"
        r"speculation:?\s*(\d+)\s*"
    )
    
    for i, section in enumerate(demo_sections):
        
        match = re.search(pattern, section, re.DOTALL | re.IGNORECASE)
        
        if match:
            title = normalize_text(match.group(1))
            explanation = normalize_text(match.group(2))
            symbol = normalize_text(match.group(3))
            category = normalize_text(match.group(4))
            speculation = normalize_text(match.group(5))
            
            matched_demo = None
            for d in demo_features:
                if d["title"] in title or title in d["title"]:
                    matched_demo = d
                    break
            if matched_demo:
                safe_key = re.sub(r"\W+", "_", matched_demo["title"])
                if prefix:
                    safe_key = f"{prefix}_{safe_key}"
                categories[f"{safe_key}_Category"] = category.strip()
                explanations[f"{safe_key}_Explanation"] = explanation.strip()
                symbols[f"{safe_key}_Symbol"] = symbol.strip()
                speculations[f"{safe_key}_Speculation"] = int(speculation.strip())

    return {**categories, **explanations, **speculations, **symbols}

# ------------------------------------------------------------
# 6. Build the request body for image inference using send_prompt
# ------------------------------------------------------------
def infer_image_demographics(image_url, model_name, demo_features):
    prompt = build_image_demo_prompt(demo_features)
    
    system_message = "You are a helpful assistant."

    try:
        result = send_prompt(
            system=system_message,
            prompt=prompt,
            image_url=image_url,
            azure_model_name=model_name,
            max_completion_tokens=16384,
            max_time=60
        )
        
        reply = result["choices"][0]["message"]["content"]
        return reply
        
    except Exception as e:
        print(f"Error in send_prompt: {e}")
        raise

# ------------------------------------------------------------
# 7. Secondary GPT cleanup function
# ------------------------------------------------------------
image_cleanup_prompt_template = """
You will be shown a long-form response to a question asking to assign a series of demographic categories to a user based solely on their profile image.
Your task is to extract and reformat the category assessments into a strict format.

CRITICAL FORMATTING RULES:
- Each demographic must be formatted EXACTLY as follows:
title: <FULL TITLE TEXT>
explanation: <FULL EXPLANATION TEXT>
symbol: <SYMBOL CODE>
category: <FULL CATEGORY TEXT>
speculation: <NUMERIC VALUE ONLY>

- Each field must be on its own separate line
- Preserve ALL original content from explanations, categories, and titles - do not summarize or shorten
- For speculation, use ONLY the numeric value (e.g., "50" not "fifty" or "50%")
- Remove any asterisks, markdown, or extra formatting from the original
- Ensure there is exactly one blank line between each complete demographic assessment
- If any field is missing, include the field name with nothing after it (e.g., "explanation:")

CHARACTER AND ENCODING RULES:
- Use only standard ASCII hyphens (-) for all dashes and hyphens in titles and categories
- Replace any special dashes (‑, –, —) with regular hyphens (-)
- Ensure all text uses standard ASCII characters only
- Remove any non-ASCII characters or convert them to their closest ASCII equivalent
- Use regular parentheses () not special parentheses

EXAMPLES OF CORRECT FORMATTING:

Example 1:
title: AGE
explanation: The profile image shows a person with visible wrinkles and gray hair...
symbol: A1)
category: 18-25
speculation: 50

Example 2:
title: SEX
explanation: The profile image presents masculine facial features and short haircut...
symbol: E1)
category: Male
speculation: 75

COMMON MISTAKES TO AVOID:
- WRONG: speculation: fifty (50)
- CORRECT: speculation: 50

- WRONG: symbol: E1) (with extra spaces)
- CORRECT: symbol: E1)

- WRONG: Using markdown like **title:**
- CORRECT: title:

- WRONG: Combining multiple fields on one line
- CORRECT: Each field on its own line

Now, clean and reformat the following response to match this exact structure. Preserve all original meaning but enforce the strict formatting and character rules:

{raw_output}
"""

def clean_image_output(raw_output, model_name):
    system_message = "You are a helpful assistant."
    
    try:
        result = send_prompt(
            system=system_message,
            prompt=image_cleanup_prompt_template.format(raw_output=raw_output),
            azure_model_name=model_name,
            max_completion_tokens=16384,
            max_time=60
        )
        
        reply = result["choices"][0]["message"]["content"]
        return reply
        
    except Exception as e:
        print(f"Error in cleanup send_prompt: {e}")
        raise

# ------------------------------------------------------------
# 8. GPT call with retry logic for image demographics
# ------------------------------------------------------------
def infer_image_demographics_with_retry(image_url, model_name, demo_features, prefix="Image_Demo", max_retries=5):
    parsed_result = {}
    response_text = None
    cleaned_text = None
    
    for attempt in range(1, max_retries + 1):
        try:
            response_text = infer_image_demographics(image_url, model_name, demo_features)
            parsed_result = parse_demo_response(response_text, demo_features, prefix=prefix)

            # if parsed result is incomplete, proceed with secondary cleanup
            if any(value is None for value in parsed_result.values()):
                print(f"[DEBUG] Parsing incomplete for image demographics → attempting cleanup.")
                cleaned_text = clean_image_output(response_text, model_name)
                parsed_result = parse_demo_response(cleaned_text, demo_features, prefix=prefix)

            # if still incomplete, raise exception to retry full GPT
            if any(value is None for value in parsed_result.values()):
                raise ValueError("Parsed result is empty. Retrying...")
            else:
                print(f"Image demographic parsing succeeded")
            return response_text, cleaned_text, parsed_result

        except Exception as e:
            print(f"Attempt {attempt}/{max_retries} failed for image demographics ({model_name}): {e}")
            if attempt < max_retries:
                time.sleep(2 ** attempt * 0.1)  
    
    return response_text, cleaned_text, parsed_result

# ------------------------------------------------------------
# 9. Main processing function
# ------------------------------------------------------------
def process_image_demographics():

    INPUT_CSV = "../generated_data/user_profiles_fixed.csv" 
    df = pd.read_csv(INPUT_CSV)
    output_file = "../generated_data/image_demographic_extraction/image_demographics_results.csv"
    
    # Initialize results
    all_results = []
    
    # Load checkpoint
    checkpoint = load_checkpoint()
    start_index = checkpoint["last_index"] + 1
    print(f"Resuming from index: {start_index}")
    
    model_name = "Llama-4-Maverick-17B-128E-Instruct-FP8"
    
    for index, row in df.iloc[start_index:].iterrows():
        try:
            print(f"Processing row {index}/{len(df)}: user_id={row.get('id', 'Unknown')}")
            
            image_url = row.get('profile_image_url', '')
            
            # Skip if no image URL available
            if pd.isna(image_url) or not str(image_url).startswith("http"):
                print(f"Skipping row {index} - no valid image URL available")
                # Create empty result structure
                result_row = {"index": index, "user_id": row.get('id', '')}
                result_row["Model"] = model_name
                result_row["Image_Demo_Raw"] = None
                result_row["Image_Demo_Cleaned"] = None
                
                # Add empty fields for all demographic features
                for feature in demo_features:
                    safe_key = re.sub(r"\W+", "_", feature["title"])
                    result_row.update({
                        f"Image_Demo_{safe_key}_Category": None,
                        f"Image_Demo_{safe_key}_Explanation": None,
                        f"Image_Demo_{safe_key}_Symbol": None,
                        f"Image_Demo_{safe_key}_Speculation": None
                    })
                
                all_results.append(result_row)
                
            else:
                # Process image demographics directly from URL
                demo_text, cleaned_demo, parsed_demo = infer_image_demographics_with_retry(
                    image_url, model_name, demo_features, prefix="Image_Demo"
                )
                
                # Build result row
                result_row = {"index": index, "user_id": row.get('id', '')}
                result_row.update({
                    "Model": model_name,
                    "Image_Demo_Raw": demo_text,
                    "Image_Demo_Cleaned": cleaned_demo
                })
                result_row.update(parsed_demo)
                
                all_results.append(result_row)
            
            # Save progress
            pd.DataFrame(all_results).to_csv(output_file, index=False, encoding="utf-8-sig")
            save_checkpoint(index)
            
            print(f"Completed row {index}")
            time.sleep(1) 
            
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            if all_results:
                pd.DataFrame(all_results).to_csv(output_file, index=False, encoding="utf-8-sig")
            continue
    
    print("Image demographic extraction complete!")
    return pd.DataFrame(all_results)

# ------------------------------------------------------------
# 10. Run the processing
# ------------------------------------------------------------
if __name__ == "__main__":
    results_df = process_image_demographics()
    print(f"Processed {len(results_df)} rows")
    print("Results saved to ../generated_data/image_demographic_extraction/image_demographics_results.csv")

Resuming from index: 3
Processing row 3/928: user_id=1449812808
Error in send_prompt: Failed to download image_url (HTTP 404): https://pbs.twimg.com/profile_images/1786570110049697792/ejocAVHa_normal.jpg
Attempt 1/5 failed for image demographics (Llama-4-Maverick-17B-128E-Instruct-FP8): Failed to download image_url (HTTP 404): https://pbs.twimg.com/profile_images/1786570110049697792/ejocAVHa_normal.jpg
Error in send_prompt: Failed to download image_url (HTTP 404): https://pbs.twimg.com/profile_images/1786570110049697792/ejocAVHa_normal.jpg
Attempt 2/5 failed for image demographics (Llama-4-Maverick-17B-128E-Instruct-FP8): Failed to download image_url (HTTP 404): https://pbs.twimg.com/profile_images/1786570110049697792/ejocAVHa_normal.jpg
Error in send_prompt: Failed to download image_url (HTTP 404): https://pbs.twimg.com/profile_images/1786570110049697792/ejocAVHa_normal.jpg
Attempt 3/5 failed for image demographics (Llama-4-Maverick-17B-128E-Instruct-FP8): Failed to download image_url